# Phase C — Measurement Layer (notebook C.6)

This notebook re-runs the Phase B baseline pipeline through the instrumented `src/`
modules and produces:

- 1 row in `results/indexing_log.csv`
- 13 rows in `results/query_log.csv`
- A new emissions block in `results/carbon/emissions.csv`

See **MASTER_PLAN.md §9** for the schemas and methodology, and the Phase C entry
in **PROGRESS_LOG.md** for the implementation decisions (Q1–Q6 and the schema
lock-in addition).

**Determinism (§13.5):** `random`, `numpy`, `torch` (if importable) are all
seeded to 42 in cell 2. Ollama options `temperature=0, seed=42` are enforced as
defaults in `src/generation.py`.

**Indexing/query separation (§13.3):** indexing-time cost and query-time cost
are wrapped in two separate CodeCarbon trackers. Per-query energy/CO2 is
proportionally allocated from the query-block tracker by per-query
`total_latency_ms`; every query row's `notes` column carries the tag
`energy_per_query: proportionally allocated from block tracker`.

**Q1 — index overwrite warning:** cell 4 rebuilds `indices/6fb92835/` to
populate the indexing measurements. The Q5(a) smoke test in cell 3 runs against
the pre-overwrite Phase B index so the prompt-refactor variable is isolated
from the rebuild variable.

In [1]:
"""Cell 2 — Imports, paths, determinism guards, run identification."""
from __future__ import annotations

import os
import random
import sys
import time
import tracemalloc
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the project root from the notebook's cwd. Jupyter sets cwd to the
# notebook directory; `jupyter lab` started from the project root keeps cwd
# at the project root. Handle both.
_HERE = Path.cwd()
ROOT = _HERE if (_HERE / "MASTER_PLAN.md").exists() else _HERE.parent
assert (ROOT / "MASTER_PLAN.md").exists(), f"Could not find project root from {_HERE}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chunking import chunk_corpus
from src.embedding import embed_chunks
from src.eval_questions import load_eval_questions
from src.indexing import build_index, load_index, save_index
from src.metrics import (
    PeakRAMSampler,
    count_retrieved_tokens,
    count_text_tokens,
    set_global_seeds,
    timing_context,
)
from src.pipeline import RAGConfig, run_rag

# §9 C.5 — determinism guards (notebook-side, in addition to the Ollama
# defaults already set in src/generation.py).
random.seed(42)
np.random.seed(42)
set_global_seeds(42)  # also seeds torch if importable

# Project paths.
RESULTS_DIR  = ROOT / "results"
CARBON_DIR   = RESULTS_DIR / "carbon"
INDICES_DIR  = ROOT / "indices"
CORPUS_TXT   = ROOT / "corpus_text"
EVAL_QS_PATH = ROOT / "dataset" / "eval_questions.md"
CARBON_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# §8 baseline config — same values as RAGConfig defaults (chunk_size=512,
# overlap_pct=10, MiniLM-L6-v2, top_k=3, llama3.1:8b-instruct-q4_K_M,
# temperature=0, seed=42).
CFG = RAGConfig()

# Run identification — shared across the indexing row and the 13 query rows.
ts_dt      = datetime.now(timezone.utc)
ts_iso     = ts_dt.isoformat(timespec="seconds")
ts_compact = ts_dt.strftime("%Y%m%dT%H%M%SZ")
RUN_ID     = f"{ts_compact}_{CFG.hash}"

print(f"[C.6] Project root: {ROOT}")
print(f"[C.6] Run ID:       {RUN_ID}")
print(f"[C.6] Config:       chunk_size={CFG.chunk_size}, overlap_pct={CFG.overlap_pct}, "
      f"embedding={CFG.embedding_model}")
print(f"[C.6] Config:       top_k={CFG.top_k}, llm={CFG.llm_model}, "
      f"temperature={CFG.temperature}, seed={CFG.seed}")

[C.6] Project root: <project_root>
[C.6] Run ID:       20260509T185155Z_6fb92835
[C.6] Config:       chunk_size=512, overlap_pct=10, embedding=sentence-transformers/all-MiniLM-L6-v2
[C.6] Config:       top_k=3, llm=llama3.1:8b-instruct-q4_K_M, temperature=0.0, seed=42


In [2]:
"""Cell 3 — Q5(a) smoke test against the pre-overwrite Phase B index.

Validates that the new build_prompt + generate_from_prompt path produces a
byte-identical answer to Phase B's results/baseline_answers.csv on the same
index. Halts the notebook on mismatch.
"""
PHASE_B_INDEX_DIR = INDICES_DIR / CFG.hash  # 6fb92835
print(f"[C.6 / Q5(a)] Smoke testing refactored generate path against "
      f"{PHASE_B_INDEX_DIR}/")
print(f"               (runs BEFORE the rebuild; isolates prompt-refactor "
      f"from rebuild)")

baseline_csv = RESULTS_DIR / "baseline_answers.csv"
assert baseline_csv.exists(), (
    f"Phase B baseline CSV missing at {baseline_csv}; cannot run Q5(a) smoke test."
)
baseline = pd.read_csv(baseline_csv)
SMOKE_QID = "Q01"
ref_row = baseline[baseline["question_id"] == SMOKE_QID]
assert len(ref_row) == 1, f"Expected exactly 1 row for {SMOKE_QID}, got {len(ref_row)}"
ref_row = ref_row.iloc[0]

idx_b, chunks_b = load_index(PHASE_B_INDEX_DIR)
print(f"               loaded Phase B index: {idx_b.ntotal} vectors, "
      f"{len(chunks_b)} chunks")

t0 = time.perf_counter()
result = run_rag(ref_row["question"], CFG, idx_b, chunks_b)
smoke_dt = time.perf_counter() - t0

got = result["answer"].strip()
expected = str(ref_row["answer"]).strip()
if got != expected:
    raise AssertionError(
        "Q5(a) smoke test FAILED — refactored generate path produced a "
        f"different answer for {SMOKE_QID}.\n"
        f"--- expected (results/baseline_answers.csv) ---\n{expected!r}\n"
        f"--- got (build_prompt + generate_from_prompt) ---\n{got!r}"
    )
print(f"[C.6 / Q5(a)] PASSED for {SMOKE_QID}: byte-identical answer "
      f"({len(got)} chars, {smoke_dt:.1f}s).")

# Free Phase B index/chunks — about to be overwritten anyway.
del idx_b, chunks_b

[C.6 / Q5(a)] Smoke testing refactored generate path against <project_root>/indices/6fb92835/
               (runs BEFORE the rebuild; isolates prompt-refactor from rebuild)
               loaded Phase B index: 754 vectors, 754 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[C.6 / Q5(a)] PASSED for Q01: byte-identical answer (814 chars, 26.5s).


In [3]:
"""Cell 4 — Indexing block (rebuild). OVERWRITES indices/6fb92835/ (Q1).

Wraps chunk_corpus + embed_chunks + build_index + save_index in:
- one CodeCarbon EmissionsTracker (project_name=phase_c_indexing)
- a timing_context() so each @time_it-decorated step records wall-clock seconds
- a PeakRAMSampler() so peak whole-process RSS is captured
- tracemalloc (Q4: Python-only allocations, printed to notebook only — NOT in CSV)
"""
from codecarbon import OfflineEmissionsTracker

INDEX_DIR = INDICES_DIR / CFG.hash  # 6fb92835 — OVERWRITES Phase B artifacts
print(f"[C.6] Rebuilding index at {INDEX_DIR} — OVERWRITES Phase B artifacts (Q1).")

idx_tracker = OfflineEmissionsTracker(
    project_name="phase_c_indexing",
    tracking_mode="machine",
    country_iso_code="DEU",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    measure_power_secs=1.0,
    log_level="error",
    save_to_file=True,
)

tracemalloc.start()
idx_tracker.start()
try:
    with timing_context() as IDX_T, PeakRAMSampler(interval_sec=0.1) as RAM:
        chunks = chunk_corpus(
            text_dir=CORPUS_TXT,
            chunk_size=CFG.chunk_size,
            overlap_pct=CFG.overlap_pct,
        )
        embeddings = embed_chunks(
            chunks,
            model_name=CFG.embedding_model,
            show_progress=False,
        )
        faiss_index = build_index(embeddings)
        save_index(faiss_index, chunks, INDEX_DIR)
finally:
    idx_co2_kg = idx_tracker.stop()
    tm_current_b, tm_peak_b = tracemalloc.get_traced_memory()
    tracemalloc.stop()

idx_energy_kwh = idx_tracker.final_emissions_data.energy_consumed
idx_energy_wh  = idx_energy_kwh * 1000.0
idx_co2_g      = idx_co2_kg     * 1000.0

# Index file size on disk.
INDEX_SIZE_MB = (INDEX_DIR / "faiss.index").stat().st_size / 1_000_000.0

# indexing_time_sec = sum of decorator-recorded sub-step times. PDF extraction
# is intentionally not measured here — corpus_text/.txt files are config-
# independent prep, not per-config indexing work (would inflate Phase D's
# 16-config sweep unfairly).
INDEXING_TIME_SEC = (
    IDX_T.get("chunking", 0.0)
    + IDX_T.get("embedding", 0.0)
    + IDX_T.get("faiss_build", 0.0)
    + IDX_T.get("index_save", 0.0)
)

print(f"[C.6] Indexing block done.")
print(f"      n_chunks_total       = {len(chunks)}")
print(f"      indexing_time_sec    = {INDEXING_TIME_SEC:.2f}")
print(f"      embedding_time_sec   = {IDX_T.get('embedding', 0.0):.2f}")
print(f"      faiss_build_time_sec = {IDX_T.get('faiss_build', 0.0):.4f}")
print(f"      index_save_time_sec  = {IDX_T.get('index_save', 0.0):.4f}")
print(f"      peak_ram_mb          = {RAM.peak_rss_mb:.1f} (delta {RAM.delta_rss_mb:.1f})")
print(f"      index_size_mb        = {INDEX_SIZE_MB:.2f}")
print(f"      energy_wh            = {idx_energy_wh:.6f}")
print(f"      co2_g                = {idx_co2_g:.6f}")
print(f"      tracemalloc peak     = {tm_peak_b / 1_000_000:.2f} MB "
      f"(Python-only — Q4: not in CSV)")

[C.6] Rebuilding index at <project_root>/indices/6fb92835 — OVERWRITES Phase B artifacts (Q1).
[C.6] Indexing block done.
      n_chunks_total       = 754
      indexing_time_sec    = 5.00
      embedding_time_sec   = 4.03
      faiss_build_time_sec = 0.0005
      index_save_time_sec  = 0.0181
      peak_ram_mb          = 660.5 (delta 88.0)
      index_size_mb        = 1.16
      energy_wh            = 0.063158
      co2_g                = 0.024060
      tracemalloc peak     = 21.26 MB (Python-only — Q4: not in CSV)


In [4]:
"""Cell 5 — Append the 1-row indexing record to results/indexing_log.csv.

Schema is verbatim from MASTER_PLAN.md §9. chunk_overlap is logged as a
PERCENTAGE (Q3: matches Phase D's experimental knob overlap_pct ∈ {0,10,20}).
"""
EXPECTED_INDEXING_COLS = [
    "run_id","timestamp","config_hash","chunk_size","chunk_overlap",
    "embedding_model","n_documents","n_chunks_total","indexing_time_sec",
    "peak_ram_mb","index_size_mb","embedding_time_sec",
    "faiss_build_time_sec","energy_wh","co2_g","notes",
]

n_documents = len({c.source_doc for c in chunks})

indexing_row = {
    "run_id":               RUN_ID,
    "timestamp":            ts_iso,
    "config_hash":          CFG.hash,
    "chunk_size":           int(CFG.chunk_size),
    "chunk_overlap":        int(CFG.overlap_pct),  # Q3: percentage, NOT tokens
    "embedding_model":      CFG.embedding_model,
    "n_documents":          int(n_documents),
    "n_chunks_total":       int(len(chunks)),
    "indexing_time_sec":    float(INDEXING_TIME_SEC),
    "peak_ram_mb":          float(RAM.peak_rss_mb),
    "index_size_mb":        float(INDEX_SIZE_MB),
    "embedding_time_sec":   float(IDX_T.get("embedding", 0.0)),
    "faiss_build_time_sec": float(IDX_T.get("faiss_build", 0.0)),
    "energy_wh":            float(idx_energy_wh),
    "co2_g":                float(idx_co2_g),
    "notes":                "",
}

# Force schema correctness by constructing the DataFrame with the canonical
# column order.
idx_row_df = pd.DataFrame([indexing_row], columns=EXPECTED_INDEXING_COLS)

idx_csv = RESULTS_DIR / "indexing_log.csv"
write_header = not idx_csv.exists()
idx_row_df.to_csv(idx_csv, mode="a", index=False, header=write_header)
print(f"[C.6] Wrote 1 row to {idx_csv} (header_written={write_header}).")

[C.6] Wrote 1 row to <project_root>/results/indexing_log.csv (header_written=True).


In [5]:
"""Cell 6 — Load the 13 evaluation questions from dataset/eval_questions.md."""
eval_questions = load_eval_questions(EVAL_QS_PATH)
print(f"[C.6] Loaded {len(eval_questions)} evaluation questions.")
type_counts = {}
for q in eval_questions:
    type_counts[q.question_type] = type_counts.get(q.question_type, 0) + 1
print(f"      by type: {type_counts}")

[C.6] Loaded 13 evaluation questions.
      by type: {'factoid': 6, 'synthesis': 3, 'numerical': 3, 'multi-doc': 1}


In [6]:
"""Cell 7 — Query block. One CodeCarbon tracker around all 13 queries (Q2).

Per-question energy/CO2 are proportionally allocated in cell 8 — they are
NOT measured per question. CodeCarbon at sub-15s windows is unreliable, and
per-question trackers would inject 1–2 s of overhead into generation_time_ms.
"""
qry_tracker = OfflineEmissionsTracker(
    project_name="phase_c_query_block",
    tracking_mode="machine",
    country_iso_code="DEU",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    measure_power_secs=1.0,
    log_level="error",
    save_to_file=True,
)

query_rows: list[dict] = []
qry_tracker.start()
try:
    for q in eval_questions:
        result = run_rag(q.question, CFG, faiss_index, chunks)
        timings = result["timings"]
        retrieved = result["retrieved_chunks"]

        embed_ms    = timings["embed"]    * 1000.0
        retrieve_ms = timings["retrieve"] * 1000.0
        generate_ms = timings["generate"] * 1000.0
        total_ms    = embed_ms + retrieve_ms + generate_ms

        retrieved_chunk_ids_str = ";".join(r.chunk.chunk_id for r in retrieved)

        query_rows.append({
            "run_id":                RUN_ID,
            "timestamp":             ts_iso,
            "config_hash":           CFG.hash,
            "question_id":           q.question_id,
            "top_k":                 int(CFG.top_k),
            "query_embed_time_ms":   float(embed_ms),
            "retrieval_time_ms":     float(retrieve_ms),
            "generation_time_ms":    float(generate_ms),
            "total_latency_ms":      float(total_ms),
            "n_retrieved_chunks":    int(len(retrieved)),
            "retrieved_token_count": int(count_retrieved_tokens(retrieved)),
            "prompt_token_count":    int(count_text_tokens(result["prompt"])),
            "answer_token_count":    int(count_text_tokens(result["answer"])),
            "energy_wh_per_query":   None,  # filled in cell 8
            "co2_g_per_query":       None,  # filled in cell 8
            "retrieved_chunk_ids":   retrieved_chunk_ids_str,
            "answer_text":           result["answer"],
            "notes":                 "",    # filled in cell 8
        })
        last = query_rows[-1]
        print(f"[C.6 / query] {q.question_id}: gen={generate_ms/1000:.1f}s "
              f"total={total_ms/1000:.1f}s tokens="
              f"r{last['retrieved_token_count']}/p{last['prompt_token_count']}/"
              f"a{last['answer_token_count']}")
finally:
    qry_co2_kg = qry_tracker.stop()

qry_energy_kwh = qry_tracker.final_emissions_data.energy_consumed
qry_energy_wh  = qry_energy_kwh * 1000.0
qry_co2_g      = qry_co2_kg     * 1000.0

print(f"[C.6] Query block done. Block totals: "
      f"energy_wh={qry_energy_wh:.6f}, co2_g={qry_co2_g:.6f}")

[C.6 / query] Q01: gen=11.2s total=11.2s tokens=r1468/p1622/a216
[C.6 / query] Q02: gen=12.6s total=12.6s tokens=r1478/p1627/a84
[C.6 / query] Q03: gen=18.7s total=18.9s tokens=r1495/p1650/a203
[C.6 / query] Q04: gen=12.1s total=12.2s tokens=r1488/p1629/a58
[C.6 / query] Q05: gen=14.6s total=14.8s tokens=r1483/p1637/a92
[C.6 / query] Q06: gen=15.9s total=16.0s tokens=r1213/p1361/a160
[C.6 / query] Q07: gen=15.1s total=15.3s tokens=r1423/p1571/a125
[C.6 / query] Q08: gen=12.2s total=12.3s tokens=r1486/p1635/a57
[C.6 / query] Q09: gen=16.8s total=16.9s tokens=r1489/p1635/a113
[C.6 / query] Q10: gen=14.3s total=14.6s tokens=r1477/p1622/a96
[C.6 / query] Q11: gen=9.7s total=9.8s tokens=r1002/p1135/a53
[C.6 / query] Q12: gen=12.1s total=12.3s tokens=r1456/p1605/a54
[C.6 / query] Q13: gen=25.5s total=25.7s tokens=r1479/p1642/a258
[C.6] Query block done. Block totals: energy_wh=2.434630, co2_g=0.927472


In [7]:
"""Cell 8 — Proportional allocation of block energy/CO2 across the 13 queries.

Weight = per-query total_latency_ms / sum(total_latency_ms). Justification: the
LLM call is ~98% of per-query wall-clock at the M4 baseline and the dominant
driver of CPU power draw during the block, so wall-clock is a defensible proxy
for energy share. The allocation rule itself is recorded in every row's notes
column (Q2 tag) so a CSV reader sees this is computed, not measured.
"""
NOTES_TAG = "energy_per_query: proportionally allocated from block tracker"

sum_latency = sum(r["total_latency_ms"] for r in query_rows)
assert sum_latency > 0, "FAIL: sum of total_latency_ms is 0 — query loop did no work?"

for r in query_rows:
    weight = r["total_latency_ms"] / sum_latency
    r["energy_wh_per_query"] = float(qry_energy_wh * weight)
    r["co2_g_per_query"]     = float(qry_co2_g     * weight)
    r["notes"]               = NOTES_TAG

print(f"[C.6] Allocated block energy/CO2 across {len(query_rows)} queries "
      f"by per-query total_latency_ms.")
print(f"      sum(weights) check = "
      f"{sum(r['energy_wh_per_query'] for r in query_rows):.6f} Wh "
      f"vs block {qry_energy_wh:.6f} Wh")

[C.6] Allocated block energy/CO2 across 13 queries by per-query total_latency_ms.
      sum(weights) check = 2.434630 Wh vs block 2.434630 Wh


In [8]:
"""Cell 9 — Append the 13-row query record to results/query_log.csv."""
EXPECTED_QUERY_COLS = [
    "run_id","timestamp","config_hash","question_id","top_k",
    "query_embed_time_ms","retrieval_time_ms","generation_time_ms",
    "total_latency_ms","n_retrieved_chunks","retrieved_token_count",
    "prompt_token_count","answer_token_count","energy_wh_per_query",
    "co2_g_per_query","retrieved_chunk_ids","answer_text","notes",
]
qry_rows_df = pd.DataFrame(query_rows, columns=EXPECTED_QUERY_COLS)

qry_csv = RESULTS_DIR / "query_log.csv"
write_header = not qry_csv.exists()
qry_rows_df.to_csv(qry_csv, mode="a", index=False, header=write_header)
print(f"[C.6] Wrote {len(qry_rows_df)} rows to {qry_csv} "
      f"(header_written={write_header}).")

[C.6] Wrote 13 rows to <project_root>/results/query_log.csv (header_written=True).


In [9]:
"""Cell 10 — Schema lock-in (Q3 follow-through) + sanity asserts (A–H, Q6).

Schema asserts run FIRST so a column-name typo halts with a clear diff before
the value asserts produce a confusing KeyError trace. On any failure: notebook
halts, CSVs persist on disk, bad rows must be removed manually before re-run.
A re-run generates a fresh RUN_ID, so the failed run's rows are NOT overwritten
— the cleanup is mandatory.
"""
EXPECTED_INDEXING_COLS = [
    "run_id","timestamp","config_hash","chunk_size","chunk_overlap",
    "embedding_model","n_documents","n_chunks_total","indexing_time_sec",
    "peak_ram_mb","index_size_mb","embedding_time_sec",
    "faiss_build_time_sec","energy_wh","co2_g","notes",
]
EXPECTED_QUERY_COLS = [
    "run_id","timestamp","config_hash","question_id","top_k",
    "query_embed_time_ms","retrieval_time_ms","generation_time_ms",
    "total_latency_ms","n_retrieved_chunks","retrieved_token_count",
    "prompt_token_count","answer_token_count","energy_wh_per_query",
    "co2_g_per_query","retrieved_chunk_ids","answer_text","notes",
]

idx_df = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
qry_df = pd.read_csv(RESULTS_DIR / "query_log.csv")

# --- Schema lock-in (runs first) ---
assert list(idx_df.columns) == EXPECTED_INDEXING_COLS, (
    f"indexing_log.csv columns drifted from §9 schema:\n"
    f"  expected: {EXPECTED_INDEXING_COLS}\n"
    f"  got:      {list(idx_df.columns)}"
)
assert list(qry_df.columns) == EXPECTED_QUERY_COLS, (
    f"query_log.csv columns drifted from §9 schema:\n"
    f"  expected: {EXPECTED_QUERY_COLS}\n"
    f"  got:      {list(qry_df.columns)}"
)
print("Schema lock-in PASSED — both CSVs match §9 schema by name and order.")

# Filter to this run only.
idx_this = idx_df[idx_df["run_id"] == RUN_ID]
qry_this = qry_df[qry_df["run_id"] == RUN_ID]

# A. Row counts.
assert len(idx_this) == 1,  f"FAIL (A): expected 1 indexing row for {RUN_ID}, got {len(idx_this)}"
assert len(qry_this) == 13, f"FAIL (A): expected 13 query rows for {RUN_ID}, got {len(qry_this)}"

# B. Run-id and config_hash linkage.
assert (qry_this["run_id"]      == RUN_ID).all(),   "FAIL (B): query rows have mismatched run_id"
assert (qry_this["config_hash"] == CFG.hash).all(), "FAIL (B): query rows have mismatched config_hash"

# C. Generation dominates retrieval (every row).
bad_gen = qry_this[qry_this["generation_time_ms"] <= qry_this["retrieval_time_ms"]]
assert bad_gen.empty, (
    "FAIL (C): generation_time_ms must be > retrieval_time_ms for every row.\n"
    f"{bad_gen[['question_id','retrieval_time_ms','generation_time_ms']]}"
)

# D. retrieved_token_count <= top_k * chunk_size * 1.10 (Q6: 10% slack for RCTS overshoot).
ceiling = int(idx_this['chunk_size'].iloc[0]) * int(qry_this['top_k'].iloc[0]) * 1.10
bad_tok = qry_this[qry_this["retrieved_token_count"] > ceiling]
assert bad_tok.empty, (
    f"FAIL (D): retrieved_token_count exceeds k*chunk_size*1.10 = {ceiling:.0f}.\n"
    f"{bad_tok[['question_id','top_k','retrieved_token_count']]}"
)

# E. CodeCarbon produced non-zero indexing energy/CO2.
assert idx_this["energy_wh"].iloc[0] > 0, "FAIL (E): indexing energy_wh is 0 — CodeCarbon misconfigured?"
assert idx_this["co2_g"].iloc[0]    > 0, "FAIL (E): indexing co2_g is 0 — CodeCarbon misconfigured?"

# F. Plausible peak RAM.
peak = float(idx_this["peak_ram_mb"].iloc[0])
assert 200 < peak < 8000, f"FAIL (F): peak_ram_mb={peak:.1f} MB outside plausible 200–8000 MB range"

# G. All answers non-empty.
empty = qry_this[qry_this["answer_text"].astype(str).str.strip() == ""]
assert empty.empty, f"FAIL (G): empty answer_text for {empty['question_id'].tolist()}"

# H. Q2 notes tag present on every query row.
NOTES_TAG = "energy_per_query: proportionally allocated from block tracker"
missing_tag = qry_this[qry_this["notes"] != NOTES_TAG]
assert missing_tag.empty, f"FAIL (H): Q2 notes tag missing on {missing_tag['question_id'].tolist()}"

print("All sanity asserts (schema lock + A–H) PASSED.")

# Final summary table.
print()
print("=" * 70)
print(f"Phase C / C.6 — Run {RUN_ID}")
print("=" * 70)
ix = idx_this.iloc[0]
print(f"Indexing:")
print(f"  chunk_size={ix.chunk_size}, chunk_overlap={ix.chunk_overlap}%, "
      f"n_chunks={ix.n_chunks_total}")
print(f"  indexing_time_sec={ix.indexing_time_sec:.2f}, "
      f"embedding_time_sec={ix.embedding_time_sec:.2f}, "
      f"faiss_build_time_sec={ix.faiss_build_time_sec:.4f}")
print(f"  peak_ram_mb={ix.peak_ram_mb:.1f}, index_size_mb={ix.index_size_mb:.2f}")
print(f"  energy_wh={ix.energy_wh:.6f}, co2_g={ix.co2_g:.6f}")
print()
print(f"Queries (n={len(qry_this)}):")
print(f"  mean generation_time_ms = {qry_this.generation_time_ms.mean():.0f} "
      f"(min {qry_this.generation_time_ms.min():.0f}, "
      f"max {qry_this.generation_time_ms.max():.0f})")
print(f"  mean total_latency_ms   = {qry_this.total_latency_ms.mean():.0f}")
print(f"  total energy_wh (block) = {qry_this.energy_wh_per_query.sum():.6f}")
print(f"  total co2_g (block)     = {qry_this.co2_g_per_query.sum():.6f}")
print(f"  mean prompt_token_count = {qry_this.prompt_token_count.mean():.0f}")
print(f"  mean answer_token_count = {qry_this.answer_token_count.mean():.0f}")

Schema lock-in PASSED — both CSVs match §9 schema by name and order.
All sanity asserts (schema lock + A–H) PASSED.

Phase C / C.6 — Run 20260509T185155Z_6fb92835
Indexing:
  chunk_size=512, chunk_overlap=10%, n_chunks=754
  indexing_time_sec=5.00, embedding_time_sec=4.03, faiss_build_time_sec=0.0005
  peak_ram_mb=660.5, index_size_mb=1.16
  energy_wh=0.063158, co2_g=0.024060

Queries (n=13):
  mean generation_time_ms = 14675 (min 9663, max 25516)
  mean total_latency_ms   = 14812
  total energy_wh (block) = 2.434630
  total co2_g (block)     = 0.927472
  mean prompt_token_count = 1567
  mean answer_token_count = 121


## Wrap-up

Phase C measurement layer is now active end-to-end:

- `results/indexing_log.csv` — the canonical indexing-time cost log. One row
  per indexing run; Phase D will append 13 more rows (one per indexed config).
- `results/query_log.csv` — the canonical query-time cost log. One row per
  (configuration, question) pair; Phase D will append 611 more rows.
- `results/carbon/emissions.csv` — CodeCarbon's per-block raw output. Two new
  blocks per Phase C/D notebook run (one indexing, one per-query loop);
  identified via the `project_name` column.

**Filter to this run** in any downstream analysis with
`df[df['run_id'] == '<RUN_ID printed in cell 10>']`. The indexing row and its
13 query rows share the same `run_id`, which is the join key for cost-versus-
quality analysis in Phase E.

**Next:** Phase D (Three Experiments). The instrumentation in this notebook
is ready to be reused as-is — Phase D notebooks just sweep `RAGConfig` over
the 9+3+4 = 16 configurations and append rows to the same two CSVs.